# Predict future procurement prices with XGBoost

This local Jupyter notebook loads the selected complete price XGBoost pipeline and `future_price_horizon_features.csv`. It validates the real future context, predicts every crop/location/H1–H12 row, and saves the resulting procurement-price forecasts under `artifacts/price/predictions`. It does not create hypothetical rows or replace missing future context with zero.

In [4]:
# Run this only if the local Jupyter environment is missing dependencies.
# %pip install -r ../../requirements.txt

## Configure local project files

The notebook detects the project root whether Jupyter starts from the repository root or a notebook folder. It checks the training-library versions before loading the serialized pipeline.

In [5]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost

def find_project_root(start_path):
    start_path = start_path.resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "pipeline_config.json").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Start Jupyter inside the demand-forecasting project."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
MODEL_FILE = PROJECT_ROOT / "artifacts" / "price" / "models" / "global_xgboost_pipeline.joblib"
NATIVE_MODEL_FILE = PROJECT_ROOT / "artifacts" / "price" / "models" / "global_xgboost_model.json"
FUTURE_DATA_FILE = PROJECT_ROOT / "data" / "processed" / "price" / "future_price_horizon_features.csv"
TRAINING_DATA_FILE = PROJECT_ROOT / "data" / "processed" / "price" / "price_horizon_training_dataset.csv"
METADATA_FILE = PROJECT_ROOT / "artifacts" / "price" / "reports" / "xgboost_training_metadata.json"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "price" / "predictions"

for required_file in [MODEL_FILE, NATIVE_MODEL_FILE, FUTURE_DATA_FILE, TRAINING_DATA_FILE, METADATA_FILE]:
    if not required_file.exists():
        raise FileNotFoundError(f"Required file not found: {required_file}")

training_metadata = json.loads(METADATA_FILE.read_text(encoding="utf-8"))
trained_versions = training_metadata.get("versions", {})
version_mismatches = {}
if trained_versions.get("scikit_learn") != sklearn.__version__:
    version_mismatches["scikit-learn"] = (trained_versions.get("scikit_learn"), sklearn.__version__)
if trained_versions.get("xgboost") != xgboost.__version__:
    version_mismatches["xgboost"] = (trained_versions.get("xgboost"), xgboost.__version__)
if version_mismatches:
    instructions = "; ".join(
        f"{package}: model={versions[0]}, local={versions[1]}"
        for package, versions in version_mismatches.items()
    )
    raise RuntimeError(
        "Training-library version mismatch. " + instructions +
        ". Install the model versions and restart the Jupyter kernel before loading."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Model: {MODEL_FILE}")
print(f"Future features: {FUTURE_DATA_FILE}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"XGBoost: {xgboost.__version__}")

Project root: C:\demand-forecasting
Model: C:\demand-forecasting\artifacts\price\models\global_xgboost_pipeline.joblib
Future features: C:\demand-forecasting\data\processed\price\future_price_horizon_features.csv
scikit-learn: 1.6.1
XGBoost: 3.4.1


## Load the pipeline and validate future features

Prediction stops if model features, future holidays, weather normals, horizons, or row keys are incomplete. This prevents plausible-looking forecasts from invalid inputs.

In [6]:
try:
    model_pipeline = joblib.load(MODEL_FILE)
    pipeline_source = "complete joblib pipeline"
except Exception as joblib_error:
    print("Complete joblib pipeline could not be loaded:", joblib_error)
    print("Using the portable native XGBoost model and rebuilding preprocessing.")

    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder
    from xgboost import XGBRegressor

    feature_columns = training_metadata["feature_columns"]
    categorical_features = training_metadata["categorical_features"]
    numeric_features = [
        column for column in feature_columns
        if column not in categorical_features
    ]
    historical_data = pd.read_csv(
        TRAINING_DATA_FILE,
        parse_dates=["forecast_origin_month", "target_month"],
    )
    final_target_month = historical_data["target_month"].max().to_period("M").to_timestamp()
    test_start = final_target_month - pd.DateOffset(months=2)
    preprocessor_fit_data = historical_data[historical_data["target_month"] < test_start]
    if preprocessor_fit_data.empty:
        raise RuntimeError("No pre-test rows are available to rebuild preprocessing.")

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])
    preprocessor = ColumnTransformer(
        [
            ("categorical", categorical_pipeline, categorical_features),
            ("numeric", numeric_pipeline, numeric_features),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    preprocessor.fit(preprocessor_fit_data[feature_columns])
    portable_model = XGBRegressor()
    portable_model.load_model(NATIVE_MODEL_FILE)
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", portable_model),
    ])
    pipeline_source = "portable JSON model + rebuilt deterministic preprocessor"

future_data = pd.read_csv(
    FUTURE_DATA_FILE,
    parse_dates=["forecast_origin_month", "target_month"],
)

feature_columns = list(training_metadata["feature_columns"])
missing_features = [column for column in feature_columns if column not in future_data.columns]
if missing_features:
    raise ValueError(f"Future price dataset is missing model features: {missing_features}")

required_future_context = [
    "target_public_holiday_count",
    "target_cny",
    "target_hari_raya",
    "target_gawai",
    "target_christmas",
    "target_major_festival",
    "target_month_normal_temperature",
    "target_month_normal_precipitation",
]
missing_context = future_data[required_future_context].isna().sum()
missing_context = missing_context[missing_context > 0]
if not missing_context.empty:
    raise ValueError(
        "Future price context is incomplete. Regenerate future_price_horizon_features.csv. "
        "Missing values: " + str(missing_context.to_dict())
    )

key_columns = ["forecast_origin_month", "crop", "location", "forecast_horizon"]
if future_data.duplicated(key_columns).any():
    raise ValueError("Future price data contains duplicate forecast keys.")
if not future_data["forecast_horizon"].between(1, 12).all():
    raise ValueError("Future price horizons must be between H1 and H12.")

expected_horizons = set(range(1, 13))
invalid_series = []
for series_key, group in future_data.groupby(["crop", "location"]):
    if set(group["forecast_horizon"]) != expected_horizons:
        invalid_series.append(series_key)
if invalid_series:
    raise ValueError(f"Some crop/location series do not contain H1–H12: {invalid_series[:10]}")

expected_rows = future_data[["crop", "location"]].drop_duplicates().shape[0] * 12
if len(future_data) != expected_rows:
    raise ValueError(f"Expected {expected_rows} H1–H12 rows but found {len(future_data)}.")

print(f"Pipeline source: {pipeline_source}")
print(f"Validated {len(future_data):,} future price rows.")
print(f"Series: {future_data[["crop", "location"]].drop_duplicates().shape[0]}")
print(f"Target range: {future_data.target_month.min().date()} to {future_data.target_month.max().date()}")

Complete joblib pipeline could not be loaded: input stream corrupted
Using the portable native XGBoost model and rebuilding preprocessing.
Pipeline source: portable JSON model + rebuilt deterministic preprocessor
Validated 540 future price rows.
Series: 45
Target range: 2026-01-01 to 2026-12-01


## Predict all crop, location, and horizon rows

In [7]:
X_future = future_data[feature_columns].copy()
predicted_price = model_pipeline.predict(X_future)
if not np.isfinite(predicted_price).all():
    raise ValueError("The model produced non-finite price predictions.")

future_data["predicted_procurement_price_rm_per_kg"] = np.maximum(predicted_price, 0)
forecast_output = future_data[
    [
        "forecast_origin_month",
        "target_month",
        "forecast_horizon",
        "crop",
        "location",
        "predicted_procurement_price_rm_per_kg",
    ]
].sort_values(["target_month", "crop", "location"]).reset_index(drop=True)

display(forecast_output.head(30))
display(
    forecast_output.groupby("target_month")["predicted_procurement_price_rm_per_kg"]
    .agg(["mean", "min", "max"])
    .round(3)
)

,forecast_origin_month,target_month,forecast_horizon,crop,location,predicted_procurement_price_rm_per_kg
0,2025-12-01,2026-01-01,1,Bittergourd,Kuching,6.090454
1,2025-12-01,2026-01-01,1,Bittergourd,Miri,6.104960
2,2025-12-01,2026-01-01,1,Bittergourd,Sibu,6.145486
3,2025-12-01,2026-01-01,1,Cabbage,Kuching,3.794209
4,2025-12-01,2026-01-01,1,Cabbage,Miri,3.756779
5,2025-12-01,2026-01-01,1,Cabbage,Sibu,3.739489
6,2025-12-01,2026-01-01,1,Choysum,Kuching,4.018752
7,2025-12-01,2026-01-01,1,Choysum,Miri,4.101424
8,2025-12-01,2026-01-01,1,Choysum,Sibu,4.052176
9,2025-12-01,2026-01-01,1,Cucumber,Kuching,2.133279


,mean,min,max
target_month,,,
2026-01-01,4.816,2.100,9.849
2026-02-01,4.865,2.132,10.013
2026-03-01,4.818,2.096,10.286
2026-04-01,4.635,2.021,9.656
2026-05-01,4.430,1.934,9.487
2026-06-01,4.481,1.961,9.792
2026-07-01,4.348,1.898,9.296
2026-08-01,4.374,1.918,9.391
2026-09-01,4.431,1.961,9.413


## Select an optional target period

The default selects the complete available future period. Replace the two dates if the application needs a smaller range.

In [8]:
SELECTED_START = forecast_output["target_month"].min()
SELECTED_END = forecast_output["target_month"].max()

selected_forecast = forecast_output[
    forecast_output["target_month"].between(SELECTED_START, SELECTED_END)
].copy()
if selected_forecast.empty:
    raise ValueError("No price forecast rows fall within the selected target range.")

print(f"Selected rows: {len(selected_forecast):,}")
display(selected_forecast.head(30))

Selected rows: 540


,forecast_origin_month,target_month,forecast_horizon,crop,location,predicted_procurement_price_rm_per_kg
0,2025-12-01,2026-01-01,1,Bittergourd,Kuching,6.090454
1,2025-12-01,2026-01-01,1,Bittergourd,Miri,6.104960
2,2025-12-01,2026-01-01,1,Bittergourd,Sibu,6.145486
3,2025-12-01,2026-01-01,1,Cabbage,Kuching,3.794209
4,2025-12-01,2026-01-01,1,Cabbage,Miri,3.756779
5,2025-12-01,2026-01-01,1,Cabbage,Sibu,3.739489
6,2025-12-01,2026-01-01,1,Choysum,Kuching,4.018752
7,2025-12-01,2026-01-01,1,Choysum,Miri,4.101424
8,2025-12-01,2026-01-01,1,Choysum,Sibu,4.052176
9,2025-12-01,2026-01-01,1,Cucumber,Kuching,2.133279


## Save full and selected price forecasts

In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
full_file = OUTPUT_DIR / "xgboost_future_price_forecast_h1_h12.csv"
selected_file = OUTPUT_DIR / (
    f"xgboost_price_forecast_{SELECTED_START:%Y_%m}_to_{SELECTED_END:%Y_%m}.csv"
)
forecast_output.to_csv(full_file, index=False)
selected_forecast.to_csv(selected_file, index=False)

print(f"Full price forecast: {full_file}")
print(f"Selected price forecast: {selected_file}")

Full price forecast: C:\demand-forecasting\artifacts\price\predictions\xgboost_future_price_forecast_h1_h12.csv
Selected price forecast: C:\demand-forecasting\artifacts\price\predictions\xgboost_price_forecast_2026_01_to_2026_12.csv
